# NYC Building Energy Benchmarking - Part A2: Data Curation & Feature Engineering
## TY B.Tech Integrated Data Warehouse, OLAP and Data Mining Project
**Domain:** Building Energy and Sustainability Analytics (NYC Local Law 84)

### Objectives:
1. Apply transparent, attribute-specific cleaning rules with full audit logging
2. Reconcile row counts and metrics before and after cleaning
3. Engineer all 9 mandatory derived attributes with documented thresholds
4. Save curated dataset to `data/curated/nyc_building_energy_curated.csv`


In [1]:
import os
import json
import numpy as np
import pandas as pd

RAW_PATH = '../data/raw/nyc_building_energy.csv'
CURATED_PATH = '../data/curated/nyc_building_energy_curated.csv'
CLEANING_LOG_PATH = '../data/staging/cleaning_log.json'

os.makedirs('../data/curated', exist_ok=True)
os.makedirs('../data/staging', exist_ok=True)

df = pd.read_csv(RAW_PATH)
initial_rows = len(df)
print(f"Loaded raw data: {initial_rows:,} rows")

cleaning_log = []
def log_step(attribute, issue, treatment, count, rationale):
    cleaning_log.append({
        'attribute': attribute,
        'issue': issue,
        'treatment': treatment,
        'affected_rows': int(count),
        'rationale': rationale
    })


Loaded raw data: 7,515 rows


### 1. Canonical Column Renaming

In [2]:
canonical_map = {
    'property_id': 'PropertyID',
    'property_name': 'PropertyName',
    'borough': 'Borough',
    'primary_property_type_self_selected': 'PrimaryPropertyType',
    'gross_floor_area_buildings_sq_ft': 'GrossFloorArea',
    'site_eui_kbtu_ft²': 'SiteEUI',
    'source_eui_kbtu_ft²': 'SourceEUI',
    'total_ghg_emissions_metric_tons_co2e': 'GHGEmissions',
    'water_use_kgal': 'WaterUse',
    'energy_star_score': 'EnergyStarScore',
    'reporting_year': 'ReportingYear',
    'bbl_10_digits_': 'BBL',
    'bin': 'BIN'
}

df.rename(columns=lambda c: canonical_map.get(c, c), inplace=True)
print("Standardized columns:", list(df.columns))


Standardized columns: ['PropertyID', 'PropertyName', 'Borough', 'PrimaryPropertyType', 'GrossFloorArea', 'SiteEUI', 'SourceEUI', 'GHGEmissions', 'WaterUse', 'EnergyStarScore', 'ReportingYear', 'BBL', 'BIN']


### 2. Missing Value & Duplicate Treatment

In [3]:
n_exact = int(df.duplicated().sum())
df.drop_duplicates(inplace=True)
log_step('All Attributes', 'Exact duplicates', 'Removed', n_exact, 'Remove identical redundant records')

n_missing_gfa = int(df['GrossFloorArea'].isnull().sum())
df = df[df['GrossFloorArea'].notnull() & (df['GrossFloorArea'] > 0)]
log_step('GrossFloorArea', 'Missing or zero floor area', 'Excluded', n_missing_gfa, 'Required for EUI and intensity normalizations')

n_missing_eui = int((df['SiteEUI'].isnull() & df['SourceEUI'].isnull()).sum())
df = df[~(df['SiteEUI'].isnull() & df['SourceEUI'].isnull())]
log_step('SiteEUI/SourceEUI', 'Both EUI metrics missing', 'Excluded', n_missing_eui, 'Cannot calculate energy performance without EUI')

cap_val = float(df['SiteEUI'].quantile(0.99))
n_capped = int((df['SiteEUI'] > cap_val).sum())
df['SiteEUI'] = np.clip(df['SiteEUI'], None, cap_val)
log_step('SiteEUI', 'Extreme measurement outlier', f'Capped at 99th pct ({cap_val:.2f})', n_capped, 'Prevent undue leverage on statistical models')

borough_fix = {'MN': 'Manhattan', 'BK': 'Brooklyn', 'QN': 'Queens', 'BX': 'Bronx', 'SI': 'Staten Island'}
df['Borough'] = df['Borough'].replace(borough_fix).str.strip().str.title()
log_step('Borough', 'Inconsistent abbreviations', 'Mapped to standard full name', len(df), 'Ensure dimensional referential integrity')

print(f"Rows after data cleaning: {len(df):,} (Removed {initial_rows - len(df):,} rows)")


Rows after data cleaning: 7,370 (Removed 145 rows)


### 3. Derived Feature Engineering (Documented Formulas)

In [4]:
def classify_size(gfa):
    if gfa < 50000: return 'Small'
    elif gfa <= 200000: return 'Medium'
    return 'Large'
df['BuildingSizeBand'] = df['GrossFloorArea'].apply(classify_size)

def classify_eui(eui):
    if eui < 50: return 'Low'
    elif eui <= 100: return 'Medium'
    elif eui <= 200: return 'High'
    return 'Very High'
df['EnergyIntensityBand'] = df['SiteEUI'].apply(classify_eui)

df['GHGIntensity'] = df['GHGEmissions'] / df['GrossFloorArea']
df['WaterIntensity'] = df['WaterUse'] / df['GrossFloorArea']

def classify_star(score):
    if pd.isna(score): return 'Unknown'
    elif score >= 75: return 'Excellent'
    elif score >= 50: return 'Good'
    elif score >= 25: return 'Fair'
    return 'Poor'
df['EnergyPerformanceBand'] = df['EnergyStarScore'].apply(classify_star)

eui_threshold_75 = float(df['SiteEUI'].quantile(0.75))
df['IsHighEnergyBuilding'] = (df['SiteEUI'] > eui_threshold_75).astype(int)
print(f"High Energy Building threshold (75th percentile): {eui_threshold_75:.2f} kBtu/sqft")

ghg_threshold_75 = float(df['GHGIntensity'].quantile(0.75))
df['IsHighGHGBuilding'] = (df['GHGIntensity'] > ghg_threshold_75).astype(int)

def map_type_group(ptype):
    ptype_str = str(ptype).lower()
    if any(k in ptype_str for k in ['multifamily', 'residential', 'apartment']): return 'Residential'
    elif 'office' in ptype_str: return 'Office'
    elif any(k in ptype_str for k in ['retail', 'store', 'supermarket']): return 'Retail'
    elif any(k in ptype_str for k in ['warehouse', 'distribution', 'industrial']): return 'Industrial'
    elif any(k in ptype_str for k in ['hospital', 'school', 'college', 'worship']): return 'Institutional'
    return 'Other'
df['PropertyTypeGroup'] = df['PrimaryPropertyType'].apply(map_type_group)
df['BoroughGroup'] = df['Borough']

print("Derived features successfully created.")


High Energy Building threshold (75th percentile): 90.91 kBtu/sqft
Derived features successfully created.


### 4. Reconciliation and Output Export

In [5]:
print("=== Pre vs Post Curation Reconciliation ===")
reconciliation = pd.DataFrame({
    'Metric': ['Total Records', 'Total Floor Area (sqft)', 'Total GHG Emissions (tCO2e)', 'Mean Site EUI (kBtu/sqft)'],
    'Pre-Curation': [initial_rows, df['GrossFloorArea'].sum(), df['GHGEmissions'].sum(), df['SiteEUI'].mean()],
    'Post-Curation': [len(df), df['GrossFloorArea'].sum(), df['GHGEmissions'].sum(), df['SiteEUI'].mean()]
})
print(reconciliation.to_string(index=False))

df.to_csv(CURATED_PATH, index=False)
with open(CLEANING_LOG_PATH, 'w') as f:
    json.dump(cleaning_log, f, indent=2)

print(f"Curated dataset saved to: {CURATED_PATH}")
print(f"Cleaning audit log saved to: {CLEANING_LOG_PATH}")


=== Pre vs Post Curation Reconciliation ===
                     Metric  Pre-Curation  Post-Curation
              Total Records  7.515000e+03   7.370000e+03
    Total Floor Area (sqft)  7.610871e+08   7.610871e+08
Total GHG Emissions (tCO2e)  4.058808e+06   4.058808e+06
  Mean Site EUI (kBtu/sqft)  8.290840e+01   8.290840e+01
Curated dataset saved to: ../data/curated/nyc_building_energy_curated.csv
Cleaning audit log saved to: ../data/staging/cleaning_log.json
